# Product Sales Analysis 2023-2024

**Objective:** Explore the dataset using pure SQL on SQLite, simulating a real-world environment where data is stored in a database.

**Business questions:**  
1. ¿Which regions, categories, and periods generate the most profit margin?
2. ¿What is the revenue projection for Q1 2025?

In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv(r'C:\Users\kumri\Downloads\Data Science Notebooks\Product Sales\product_sales_dataset_final.csv')
db_path = 'sales.db'
print('Shape:',df.shape)
print('Columns:',df.columns.tolist())
df.head(3)

Shape: (200000, 14)
Columns: ['Order_ID', 'Order_Date', 'Customer_Name', 'City', 'State', 'Region', 'Country', 'Category', 'Sub_Category', 'Product_Name', 'Quantity', ' Unit_Price ', ' Revenue ', ' Profit ']


,Order_ID,Order_Date,Customer_Name,City,State,Region,Country,Category,Sub_Category,Product_Name,Quantity,Unit_Price,Revenue,Profit
0,1,08-23-23,Bianca Brown,Jackson,Mississippi,South,United States,Accessories,Small Electronics,Phone Case,3,201.01,603.03,221.49
1,2,12-20-24,Jared Edwards,Grand Rapids,Michigan,Centre,United States,Accessories,Small Electronics,Charging Cable,4,74.30,297.20,97.09
2,3,01-29-24,Susan Valdez,Minneapolis,Minnesota,Centre,United States,Clothing & Apparel,Sportswear,Nike Air Force 1,1,68.19,68.19,25.47


In [3]:
#Normalize column names
df.columns = (df.columns
    .str.strip()
    .str.lower()
    .str.replace('-','_')
    .str.replace(' ','_')
    
)

#SQLite3
conn = sqlite3.connect(db_path)
df.to_sql('sales', conn, if_exists='replace', index=False)
print(f'Table "Sales" created with {len(df):,} lines in {db_path}')
conn.close()

Table "Sales" created with 200,000 lines in sales.db


In [4]:
#Query to retrun dataframe
with sqlite3.connect(db_path) as conn:
    conn.execute("""
        UPDATE sales
        SET order_date = '20' ||SUBSTR(order_date,7,2)||'-'
                              ||SUBSTR(order_date,1,2)||'-'
                              ||SUBSTR(order_date,4,2)
        WHERE LENGTH(order_date) = 8 
    """)
    conn.commit()
print("Fixed Dates")

Fixed Dates


## 1. Data Understanding

The dataset contains 200,000 rows representing individual sales transactions. A null-value check was performed across all key columns, returning no missing data. The available categories, sub-categories, and regions by transactions were catalogued to understand the business dimensions before any analysis.

In [5]:
#Query to retrun dataframe
def query(sql:str):
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(sql, conn)

In [6]:
#Total count and date range 
query("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT customer_name) AS unique_customers,
    COUNT(DISTINCT product_name) AS unique_products,
    MIN(order_date) as first_date,
    MAX(order_date) as last_date
FROM sales
""")

,total_rows,unique_orders,unique_customers,unique_products,first_date,last_date
0,200000,200000,120230,49,2023-01-01,2024-12-31


In [7]:
#Geting column names
query("""pragma table_info(sales)""")

,cid,name,type,notnull,dflt_value,pk
0,0,order_id,INTEGER,0,None,0
1,1,order_date,TEXT,0,None,0
2,2,customer_name,TEXT,0,None,0
3,3,city,TEXT,0,None,0
4,4,state,TEXT,0,None,0
5,5,region,TEXT,0,None,0
6,6,country,TEXT,0,None,0
7,7,category,TEXT,0,None,0
8,8,sub_category,TEXT,0,None,0
9,9,product_name,TEXT,0,None,0


In [8]:
#Seaching for null values
query("""
SELECT
SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_date,
SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_region,
SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS null_category,
SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue,
SUM(CASE WHEN profit IS NULL THEN 1 ELSE 0 END) AS null_profit
FROM sales
""")

,null_order_id,null_date,null_region,null_category,null_revenue,null_profit
0,0,0,0,0,0,0


In [9]:
#Categories availables
query("""
SELECT
category,
count(distinct sub_category) as num_subcategories,
group_concat(distinct sub_category) as subcategories
from sales
group by category
order by category""")

,category,num_subcategories,subcategories
0,Accessories,3,"Small Electronics,Bags,Wearable Accessories"
1,Clothing & Apparel,5,"Sportswear,Men's Wear,Footwear,Women's Wear,Ki..."
2,Electronics,6,"Home Appliances,Wearables,Smartphones,Tablets,..."
3,Home & Furniture,5,"Storage,Bedding,Home Decor,Kitchenware,Furniture"


In [10]:
#Regions
query("""
select region, count(*) as transactions
from sales
group by region
order by transactions desc
""")

,region,transactions
0,East,57034
1,West,55428
2,Centre,49603
3,South,37935


## 2. Global KPIs

This were calculated across the entire dataset: total revenue, total profit, total units sold, average revenue per order, and overall profit margin percentage. They were boken down by year to have a good comparison.

In [20]:
#KPIs per year
query("""
select
strftime('%Y', order_date) as year,
round(sum(revenue),2) as total_revenue,
round(sum(profit),2) as total_profit,
sum(quantity) as total_units,
round(sum(profit)/sum(revenue)*100,2) as profit_margin,
count(*) as transactions
from sales
group by year
order by year
""")

,year,total_revenue,total_profit,total_units,profit_margin,transactions
0,2023,70755372.66,15678961.99,184609,22.16,99807
1,2024,71652372.27,15869646.14,186191,22.15,100193


## 3. Analysis by region and category

Revenue and profit margin were analyzed at two levels. First, grouped by region to identify which geographic areas drive the most revenue. Second, grouped by category and sub-category to surface which product lines generate the highest profit margins, revealing where the business is most and least profitable at the product level.

In [28]:
#Revenue by margin and region
query("""
select 
region,
round(sum(revenue),2) as total_revenue,
round(sum(profit),2) as total_profit,
round(sum(profit)/sum(revenue)*100,2) as profit_margin,
count(*) as transactions
from sales
group by region
order by profit_margin desc
""")

,region,total_revenue,total_profit,profit_margin,transactions
0,South,25102960.64,5918454.17,23.58,37935
1,West,36242841.73,8313962.76,22.94,55428
2,Centre,36081894.34,8094863.77,22.43,49603
3,East,44980048.22,9221327.43,20.50,57034


In [14]:
#Profit margin per subcategory
query("""
select
category,
sub_category,
round(sum(revenue),2) as total_revenue,
round(sum(profit),2) as total_profit,
round(sum(profit)/sum(revenue)*100,2) as profit_margin
from sales
group by category, sub_category
having sum(revenue)>0
order by profit_margin desc
""")

,category,sub_category,total_revenue,total_profit,profit_margin
0,Accessories,Bags,3186076.31,1087133.83,34.12
1,Accessories,Wearable Accessories,3180625.91,1080926.02,33.98
2,Accessories,Small Electronics,3746552.39,1269986.43,33.90
3,Clothing & Apparel,Kids Wear,4931503.91,1607783.49,32.60
4,Clothing & Apparel,Sportswear,5813495.93,1895311.07,32.60
5,Clothing & Apparel,Women's Wear,5327487.62,1732455.92,32.52
6,Clothing & Apparel,Footwear,5421641.72,1762753.64,32.51
7,Clothing & Apparel,Men's Wear,5640236.12,1828547.37,32.42
8,Home & Furniture,Kitchenware,9117341.17,2150176.97,23.58
9,Home & Furniture,Storage,7746306.75,1823643.82,23.54


## 4. Ranking and growth

RANK() OVER PARTITION BY was applied to rank regions by total revenue within each year, using a subquery as the data source. 
  
Month-over-month revenue growth was calculated using LAG(), which retrieves the previous row's value.

The top 10 products by total revenue were retrieved, including their profit margin and total units sold across both years.

In [15]:
#Rank of regions by revenue per year
query("""
select
    year,
    region,
    total_revenue,
    rank()over(partition by year order by total_revenue desc) as rank_in_year
from(
    select
        strftime('%Y', order_date) as year,
        region,
        round(sum(revenue),2) as total_revenue
    from sales
    group by year, region
)
order by year, rank_in_year
""")

,year,region,total_revenue,rank_in_year
0,2023,East,22465944.57,1
1,2023,Centre,17994522.22,2
2,2023,West,17943390.37,3
3,2023,South,12351515.50,4
4,2024,East,22514103.65,1
5,2024,West,18299451.36,2
6,2024,Centre,18087372.12,3
7,2024,South,12751445.14,4


In [16]:
#Month over month growth with LAG()
query("""
with monthly as (
    select 
        strftime('%Y-%m', order_date) as month,
        round(sum(revenue),2) as monthly_revenue
    from sales
    group by month
)
select
    month,
    monthly_revenue,
    LAG(monthly_revenue) over (order by month) as prev_month_revenue,
    round(
        100*(monthly_revenue - LAG(monthly_revenue) over (order by month))
        /LAG(monthly_revenue) over (order by month),
        2) as monthly_growth_pct
from monthly
order by month desc
""")


,month,monthly_revenue,prev_month_revenue,monthly_growth_pct
0,2024-12,10134372.67,13899671.91,-27.09
1,2024-11,13899671.91,8719541.99,59.41
2,2024-10,8719541.99,4846454.51,79.92
3,2024-09,4846454.51,4429328.35,9.42
4,2024-08,4429328.35,4367652.54,1.41
5,2024-07,4367652.54,4721684.53,-7.50
6,2024-06,4721684.53,4883303.43,-3.31
7,2024-05,4883303.43,4263479.24,14.54
8,2024-04,4263479.24,4088588.81,4.28
9,2024-03,4088588.81,2930941.45,39.50


In [17]:
#Top 10 products by total revenue
query("""
select
    product_name,
    category,
    round(sum(revenue),2) as total_revenue,
    round(sum(profit)/sum(revenue)*100,2) as profit_margin_pct,
    sum(quantity) as units_sold
from sales
group by product_name, category
order by total_revenue desc
limit 10
""")

,product_name,category,total_revenue,profit_margin_pct,units_sold
0,Tempur-Pedic Mattress,Home & Furniture,9061755.86,23.55,10333
1,MacBook Air,Electronics,7362516.81,13.97,8763
2,Apple Watch,Electronics,6834472.35,14.02,12368
3,Apple iPhone 14,Electronics,5740819.18,14.09,7642
4,iPad Pro,Electronics,5574458.89,14.02,10075
5,Instant Pot,Home & Furniture,4566552.55,23.76,10024
6,Storage Rack,Home & Furniture,4463941.27,23.57,9394
7,Instant Pot,Electronics,4336922.71,13.95,8176
8,Brooklinen Sheets,Home & Furniture,3981027.44,23.46,8242
9,Samsung Galaxy S23,Electronics,3851240.02,14.09,6974


In [18]:
#Saving csv
df_monthly = query("""
select
strftime('%Y-%m', order_date) as month,
round(sum(revenue),2) as monthly_revenue,
round(sum(profit),2) as monthly_profit
from sales
group by month
order by month
""")
df_monthly.to_csv(r'C:\Users\kumri\Downloads\Data Science Notebooks\Product Sales\monthly_timeseries.csv', 
                  index=False)

df_clean = query("select * from sales")
df_clean.to_csv(r'C:\Users\kumri\Downloads\Data Science Notebooks\Product Sales\sales_clean.csv', 
                index=False)
print('Dataframes saved!')

Dataframes saved!
